In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "src").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

In [2]:
import torch
from src.models.vit_mae import PatchEmbed

dummy = torch.randn(4, 1, 16, 4096)   # fake batch of 4 frames
pe = PatchEmbed()
out = pe(dummy)
print("input:", dummy.shape)
print("patches:", out.shape)

input: torch.Size([4, 1, 16, 4096])
patches: torch.Size([4, 256, 256])


In [3]:
from src.models.vit_mae import *

x_kept, mask, ids_restore = random_masking(out)   # out is (4,256,256) from before
print("kept patches:", x_kept.shape)
print("mask shape:", mask.shape, "| hidden per frame:", mask[0].sum().item())

kept patches: torch.Size([4, 64, 256])
mask shape: torch.Size([4, 256]) | hidden per frame: 192.0


In [4]:
import importlib, src.models.vit_mae
importlib.reload(src.models.vit_mae)
from src.models.vit_mae import ViTMAE

model = ViTMAE()
imgs = torch.randn(4, 1, 16, 4096)
loss, pred, mask = model(imgs)
print("loss:", loss.item())
print("pred shape:", pred.shape)
n_params = sum(p.numel() for p in model.parameters())
print("params:", f"{n_params/1e6:.1f}M")

loss: 1.3278309106826782
pred shape: torch.Size([4, 256, 256])
params: 6.7M


In [5]:
import numpy as np

frame = np.load(repo_root / "data/synthetic/quiet/quiet_00000.npy").astype(np.float32)
frame = (frame - frame.mean()) / (frame.std() + 1e-8)
frame = torch.from_numpy(frame).unsqueeze(0).unsqueeze(0)   # -> (1,1,16,4096)

loss, pred, mask = model(frame)
print("real-frame loss:", loss.item(), "| pred:", pred.shape)

real-frame loss: 1.3192483186721802 | pred: torch.Size([1, 256, 256])


In [6]:
import torch

print("MPS available:", torch.backends.mps.is_available())
print("CUDA available:", torch.cuda.is_available())

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("using device:", device)

MPS available: True
CUDA available: False
using device: mps
